## Nivel 2: Fechas 

Las fechas aportan un valor analítico muy importante porque revelan tendencias, estacionalidad y cohortes, sin embargo, suelen ser los datos más problemas en su formato.

Para este nivel haremos una muestra ya que necesitamos emplear operaciones mas complejas que pueden agotar el kernel, ya que la memoria RAM requiere mucho esfuerzo para ejecutar operaciones para el data set completo.

Para este data set tenemos started_at y ended_at y haremos el EDA en tres momentos

- Data Profiling (Validación estructural de fechas): formato consistente, longitud, rango, nulos, duplicados esperables
- Type Casting Conversión técnica de String a Datetime64.
- Business Rules Validation (Coherencia temporal): inicio ≤ fin, duración válida, detección de outliers

In [4]:
#muestra

import pandas as pd 

df=pd.read_csv('dataset_maestro_ciclistas.csv')
df_muestra = df.sample(n=100000, random_state=42)

In [5]:
#df_muestra.to_csv('muestra_ciclistas.csv', index=False)

In [6]:
df_muestra.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
2021231,AC09C4230F16DC5B,electric_bike,2025-06-30 19:16:08.604,2025-06-30 19:21:26.124,Wolcott Ave & Fargo Ave,CHI01266,Elmwood Ave & Austin St,CHI00739,42.016977,-87.677725,42.025784,-87.684107,casual
1103055,A38E8273171D67F0,electric_bike,2025-05-18 12:36:15.975,2025-05-18 12:55:40.536,NaN,NaN,Montrose Harbor,TA1308000012,41.900000,-87.620000,41.963982,-87.638181,member
1834169,0B1F607D321397B0,electric_bike,2025-06-17 20:43:05.503,2025-06-17 21:04:51.961,NaN,NaN,Michigan Ave & Oak St,CHI00252,41.950000,-87.650000,41.900960,-87.623777,member
277055,21649CA95A4A10B4,electric_bike,2025-02-02 05:04:52.086,2025-02-02 05:08:16.654,NaN,NaN,Clark St & Newport St,632,41.950000,-87.670000,41.944540,-87.654678,casual
740603,D6CB6A805FC24CD2,classic_bike,2025-04-07 10:24:51.016,2025-04-07 10:31:27.276,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member


In [7]:
df_muestra.shape

(100000, 13)

In [8]:
df_muestra[['started_at','ended_at']].head(10)

,started_at,ended_at
2021231,2025-06-30 19:16:08.604,2025-06-30 19:21:26.124
1103055,2025-05-18 12:36:15.975,2025-05-18 12:55:40.536
1834169,2025-06-17 20:43:05.503,2025-06-17 21:04:51.961
277055,2025-02-02 05:04:52.086,2025-02-02 05:08:16.654
740603,2025-04-07 10:24:51.016,2025-04-07 10:31:27.276
1718938,2025-06-26 20:38:46.166,2025-06-26 21:08:31.115
396011,2025-03-03 08:06:48.805,2025-03-03 08:22:54.075
3747332,2025-09-28 08:51:17.999,2025-09-28 09:22:02.975
1960416,2025-06-28 16:18:25.851,2025-06-28 16:51:47.192
3198153,2025-08-11 08:15:48.662,2025-08-11 08:49:14.477


In [9]:
#tipos de datos de las columnas de fecha y hora

df_muestra['ended_at'].apply(type).value_counts()
df_muestra['started_at'].apply(type).value_counts()

#Acción pendiente: convertir las columnas de fecha y hora a formato datetime

started_at
<class 'str'>    100000
Name: count, dtype: int64

In [10]:
#Longitud de los valores en 'started_at' y 'ended_at'

df_muestra['started_at'].astype('str').str.len().value_counts()
df_muestra['ended_at'].astype('str').str.len().value_counts()

ended_at
23    100000
Name: count, dtype: int64

In [11]:
# Inspección visual de los separadores únicos en 'started_at' y 'ended_at' 

print(df_muestra['started_at'].replace(r'\d',' ', regex=True).unique())
print(df_muestra['ended_at'].replace(r'\d',' ', regex=True).unique())

['    -  -     :  :  .   ']
['    -  -     :  :  .   ']


In [12]:
#Verificar que el rango de fechas en 'started_at' y 'ended_at' sea coherente y no contenga valores atípicos o inconsistentes.

print(df_muestra[['started_at','ended_at']].max())
print(df_muestra[['started_at','ended_at']].min())

started_at    2025-12-31 23:46:07.383
ended_at      2025-12-31 23:56:50.685
dtype: object
started_at    2024-12-31 23:42:36.959
ended_at      2025-01-01 00:16:16.440
dtype: object


In [13]:
#Extraer componentes de fecha y hora de 'started_at' para comprbar que corresponden a AAA/MM/DD HH:MM:SS    

partes_fecha=df_muestra['started_at'].str.split('-|:| ', expand=True)
partes_fecha.head()

,0,1,2,3,4,5
2021231,2025,06,30,19,16,08.604
1103055,2025,05,18,12,36,15.975
1834169,2025,06,17,20,43,05.503
277055,2025,02,02,05,04,52.086
740603,2025,04,07,10,24,51.016


In [14]:
# Extracción de año, mes y día de 'started_at' para saber que valores únicos hay

partes_fecha_s=df_muestra['started_at'].str.split('-|:| ', expand=True)
año_s=partes_fecha_s[0].unique()
mes_s=partes_fecha_s[1].unique()
dia_s=sorted(partes_fecha_s[2].unique())

año_s, mes_s, dia_s

(array(['2025', '2024'], dtype=object),
 array(['06', '05', '02', '04', '03', '09', '08', '07', '01', '11', '10',
        '12'], dtype=object),
 ['01',
  '02',
  '03',
  '04',
  '05',
  '06',
  '07',
  '08',
  '09',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17',
  '18',
  '19',
  '20',
  '21',
  '22',
  '23',
  '24',
  '25',
  '26',
  '27',
  '28',
  '29',
  '30',
  '31'])

In [15]:
# Extracción de año, mes y día de 'ended_at' para saber que valores únicos hay

partes_fecha_e=df_muestra['ended_at'].str.split('-|:| ', expand=True)
año_e=partes_fecha_e[0].unique()
mes_e=partes_fecha_e[1].unique()
dia_e=sorted(partes_fecha_e[2].unique())

año_e, mes_e, dia_e


(array(['2025'], dtype=object),
 array(['06', '05', '02', '04', '03', '09', '08', '07', '01', '11', '10',
        '12'], dtype=object),
 ['01',
  '02',
  '03',
  '04',
  '05',
  '06',
  '07',
  '08',
  '09',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17',
  '18',
  '19',
  '20',
  '21',
  '22',
  '23',
  '24',
  '25',
  '26',
  '27',
  '28',
  '29',
  '30',
  '31'])

In [16]:
#Verificar si hay valores nulos en 'started_at' y 'ended_at'

df_muestra['started_at'].isna().sum()
df_muestra['ended_at'].isna().sum()

np.int64(0)

In [17]:
#Revisamos duplicados, aunque es logico que existan duplicados en fechas de inicio y fin de viajes, pero sin que la cifra sea exagerada
#Además estos duplicados se conservan ya que los identificadores de viaje son únicos, entonces no afectan la unicidad del registro.

print(df_muestra['started_at'].duplicated().sum())
print(df_muestra['ended_at'].duplicated().sum())


0
0


### Data profiling completo

Se verificó que no hay nulos, las fechas tienen longitud y formato coherente para todos los registros de started_at y ended_at, hay duplicados válidos.

In [18]:
# Convertir fechas a formato datetime

df_muestra['started_at']=pd.to_datetime(df_muestra['started_at'], errors='coerce')
df_muestra['ended_at']=pd.to_datetime(df_muestra['ended_at'], errors='coerce')

In [19]:
#Revisar que la conversión a datetime se haya realizado correctamente y que no haya valores nulos introducidos por errores de formato.

print(df_muestra[['started_at','ended_at']].head())
print(df_muestra[['started_at','ended_at']].dtypes)
print(df_muestra['started_at'].isna().sum())
print(df_muestra['ended_at'].isna().sum())
print(df_muestra[['started_at','ended_at']].shape)

                     started_at                ended_at
2021231 2025-06-30 19:16:08.604 2025-06-30 19:21:26.124
1103055 2025-05-18 12:36:15.975 2025-05-18 12:55:40.536
1834169 2025-06-17 20:43:05.503 2025-06-17 21:04:51.961
277055  2025-02-02 05:04:52.086 2025-02-02 05:08:16.654
740603  2025-04-07 10:24:51.016 2025-04-07 10:31:27.276
started_at    datetime64[ns]
ended_at      datetime64[ns]
dtype: object
0
0
(100000, 2)


### Type Casting Completo

Las fechas fueron convertidas de string a datetime64[ns]. Además se hizo una validación rapida, confirmando nuevamente que no hay nulos, y las filas estan completas.
Aspectos a revisar duracion del viaje: que dure  un tiempo logico, no -4 minutos o 00 minutos. Duraciones extremas: minimo y maximo de fechas. Media mediana y moda


In [20]:
#Orden temporal de las fechas

orden_invalido=df_muestra['started_at'] > df_muestra['ended_at']
print(orden_invalido.sum())
df_muestra[orden_invalido].head()

2


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
5107759,083534D28DA37F72,classic_bike,2025-11-02 01:17:57.001,2025-11-02 01:05:27.752,Clark St & Grace St,CHI00301,Grace St & Central Ave,CHI01799,41.95078,-87.659172,41.949533,-87.767265,member
5333210,F87FA50B8D40FA97,electric_bike,2025-11-02 01:54:25.185,2025-11-02 01:10:50.859,NaN,NaN,Michigan Ave & Madison St,CHI01915,41.88000,-87.640000,41.882134,-87.625125,casual


In [21]:
#Calcular la duración de los viajes en minutos y verificar si hay valores negativos, lo cual indicaría un error en las fechas.

df_muestra['duration_min']=(df_muestra['ended_at'] - df_muestra['started_at']).dt.total_seconds() / 60
tiempo_negativo= df_muestra['duration_min']<0
print(tiempo_negativo.sum())    
tiempo_negativo.value_counts()

#El resultado muestra que los registros con duración negativa son 2 corresponde al orden temporal inválido que se identificó previamente, por lo que se confirma que esos registros tienen un error en las fechas de inicio y fin del viaje.

2


duration_min
False    99998
True         2
Name: count, dtype: int64

In [22]:
#Verificar si hay registros con duración de viaje igual a cero, lo cual podría indicar viajes extremadamente cortos o errores en las fechas.

tiempo_cero= df_muestra['duration_min']==0
print(tiempo_cero.sum())

0


In [23]:
#Revisar el rango de duración de los viajes para identificar posibles valores atípicos o errores en los datos.

print(df_muestra['duration_min'].min())
print(df_muestra['duration_min'].max())

-43.5721
1499.9600500000001


In [24]:
#Estoy analizando los percentiles para entender mejor la distribución de la duración de los viajes y detectar posibles valores atípicos.

df_muestra['duration_min'].describe(percentiles=[0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99])

#Definitivamente un valor de 25 horas es atipico ya que incluso el 99% de los viajes duran menos o igual a (1h 8min-88.70 min).

count    100000.000000
mean         15.978853
std          54.650981
min         -43.572100
1%            0.258716
5%            2.067148
10%           3.174220
25%           5.372471
50%           9.437192
75%          16.598817
90%          28.362220
95%          39.825555
99%          88.704491
max        1499.960050
Name: duration_min, dtype: float64

In [25]:
#Inspeccionde outliers en la duración de los viajes
outlier_2h=(df_muestra['duration_min']>2*60).sum()
outlier_6h=(df_muestra['duration_min']>6*60).sum()      
outlier_12h=(df_muestra['duration_min']>12*60).sum()
outlier_24h=(df_muestra['duration_min']>24*60).sum()

print('Outlier mayor a dos horas: ', outlier_2h)
print('Outlier mayor a seis horas: ', outlier_6h)           
print('Outlier mayor a doce horas: ', outlier_12h)
print('Outlier mayor a veinticuatro horas: ', outlier_24h)

Outlier mayor a dos horas:  561
Outlier mayor a seis horas:  168
Outlier mayor a doce horas:  131
Outlier mayor a veinticuatro horas:  100


In [26]:
#Inspección de los outliers en la duración de los viajes, Se crean columnas booleanas para identificar los outliers

df_muestra["outlier_2h"]=(df_muestra['duration_min']>2*60)
df_muestra["outlier_6h"]=(df_muestra['duration_min']>6*60)
df_muestra["outlier_12h"]=(df_muestra['duration_min']>12*60)       
df_muestra["outlier_24h"]=(df_muestra['duration_min']>24*60)
print('Outlier mayor a dos horas: ', df_muestra["outlier_2h"].sum())
print('Outlier mayor a seis horas: ', df_muestra["outlier_6h"].sum())
print('Outlier mayor a doce horas: ', df_muestra["outlier_12h"].sum())
print('Outlier mayor a veinticuatro horas: ', df_muestra["outlier_24h"].sum())

Outlier mayor a dos horas:  561
Outlier mayor a seis horas:  168
Outlier mayor a doce horas:  131
Outlier mayor a veinticuatro horas:  100


In [27]:
# Fechas fuera del rango esperado

print("Fechas maximas de started_at y ended_at: ", df_muestra['started_at'].max(), df_muestra['ended_at'].max())
print("Fechas minimas de started_at y ended_at: ", df_muestra['started_at'].min(), df_muestra['ended_at'].min())

Fechas maximas de started_at y ended_at:  2025-12-31 23:46:07.383000 2025-12-31 23:56:50.685000
Fechas minimas de started_at y ended_at:  2024-12-31 23:42:36.959000 2025-01-01 00:16:16.440000


In [28]:
# Relación entre duración del viaje y tipo de usuario

print(df_muestra.groupby("member_casual")["outlier_2h"].mean()*100)
print(df_muestra.groupby("member_casual")["outlier_6h"].mean()*100)
print(df_muestra.groupby("member_casual")["outlier_12h"].mean()*100)
print(df_muestra.groupby("member_casual")["outlier_24h"].mean()*100)


member_casual
casual    1.171724
member    0.214686
Name: outlier_2h, dtype: float64
member_casual
casual    0.326093
member    0.078353
Name: outlier_6h, dtype: float64
member_casual
casual    0.270823
member    0.051713
Name: outlier_12h, dtype: float64
member_casual
casual    0.212789
member    0.036042
Name: outlier_24h, dtype: float64


In [29]:
# Relacion entre duración del viaje y tipo de bicicleta

print(df_muestra.groupby('rideable_type')['outlier_2h'].mean()*100)
print(df_muestra.groupby('rideable_type')['outlier_6h'].mean()*100)
print(df_muestra.groupby('rideable_type')['outlier_12h'].mean()*100)
print(df_muestra.groupby('rideable_type')['outlier_24h'].mean()*100)

rideable_type
classic_bike     1.243937
electric_bike    0.192456
Name: outlier_2h, dtype: float64
rideable_type
classic_bike     0.456491
electric_bike    0.012317
Name: outlier_6h, dtype: float64
rideable_type
classic_bike     0.373752
electric_bike    0.000000
Name: outlier_12h, dtype: float64
rideable_type
classic_bike     0.285307
electric_bike    0.000000
Name: outlier_24h, dtype: float64


Se encontro lo siguiente: 2 registros que no tienen un orden valido, es decir, la hora de inicio de ruta es mayor a la hora de finalización, por lo tanto eso genera que hayan 2 duraciones negativas, no hay duraciones en 0, el promedio de viajes son 16 minutos, el 99% de los datos estan dentro del tiempo estimado de 88.70 minutos. El tiempo maximo de un viaje es de 25 horas y existen outliers de 2 horas en adelante. Al hacer la relacion con el tipo de bicicleta y el tipo de usuario se encontro que los outliers se encuentran especialmente en miembros casuales y bicicletas clasicas. Es necesario decidir qué hacer al respecto, cómo limpiarlos


 (Duda: osea una vez que limpie estos archivos ya no apareceran los datos sucios, sera que es buena practica dejar los datos sucios en una variable, con fines de trazabilidad, o no? porque igual eso consume recursos, En el mundo real que es lo que sucede o cuales son las buenas practicas)

### Business Rules Validation Completo

La revisión que se hizo en este paso incluye: started_at y ended_at convertidas a datetime, No nulos en fechas, Rango de fechas coherente (min/max), Regla lógica: ended_at >= started_at, Duración no negativa, Duración no extrema (outliers)

Se encontró lo siguiente:

- 2 registros con orden temporal invertido → duración negativa

- Duración máxima: 25 horas

- Los outliers se concentran en miembros casuales y bicicletas clasicas:

### Business Rules Validation Completo

La revisión que se hizo en este paso incluye: started_at y ended_at convertidas a datetime, No nulos en fechas, Rango de fechas coherente (min/max), Regla lógica: ended_at >= started_at, Duración no negativa, Duración no extrema (outliers)

Se encontró lo siguiente:

- 2 registros con orden temporal invertido → duración negativa

- Duración máxima: 25 horas

- Los outliers se concentran en miembros casuales y bicicletas clasicas:

Outlier mayor a dos horas:  561

Outlier mayor a seis horas:  168

Outlier mayor a doce horas:  131

Outlier mayor a veinticuatro horas:  100



In [30]:
# se crean variables con los datos limpios de duración del viaje
# esto con el fin de evaluar cómo se afectarás las metricas con y sin outliers

tiempo_valido= df_muestra[df_muestra['duration_min']>=0]
tiempo_valido.shape
tiempo_valido

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_min,outlier_2h,outlier_6h,outlier_12h,outlier_24h
2021231,AC09C4230F16DC5B,electric_bike,2025-06-30 19:16:08.604,2025-06-30 19:21:26.124,Wolcott Ave & Fargo Ave,CHI01266,Elmwood Ave & Austin St,CHI00739,42.016977,-87.677725,42.025784,-87.684107,casual,5.292000,False,False,False,False
1103055,A38E8273171D67F0,electric_bike,2025-05-18 12:36:15.975,2025-05-18 12:55:40.536,NaN,NaN,Montrose Harbor,TA1308000012,41.900000,-87.620000,41.963982,-87.638181,member,19.409350,False,False,False,False
1834169,0B1F607D321397B0,electric_bike,2025-06-17 20:43:05.503,2025-06-17 21:04:51.961,NaN,NaN,Michigan Ave & Oak St,CHI00252,41.950000,-87.650000,41.900960,-87.623777,member,21.774300,False,False,False,False
277055,21649CA95A4A10B4,electric_bike,2025-02-02 05:04:52.086,2025-02-02 05:08:16.654,NaN,NaN,Clark St & Newport St,632,41.950000,-87.670000,41.944540,-87.654678,casual,3.409467,False,False,False,False
740603,D6CB6A805FC24CD2,classic_bike,2025-04-07 10:24:51.016,2025-04-07 10:31:27.276,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,6.604333,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3212739,886300CA00A48C3E,electric_bike,2025-08-29 22:35:13.187,2025-08-29 22:46:34.353,NaN,NaN,Indiana Ave & Roosevelt Rd,CHI00450,41.860000,-87.620000,41.867888,-87.623041,casual,11.352767,False,False,False,False
4688668,C90B7D75B09195D9,electric_bike,2025-10-31 08:52:08.332,2025-10-31 08:56:15.942,Morgan Ave & 14th Pl,CHI00261,Throop St & Taylor St,CHI00389,41.862378,-87.651062,41.868968,-87.659141,member,4.126833,False,False,False,False
4958468,725DA042D4D32887,electric_bike,2025-10-14 16:27:10.291,2025-10-14 16:47:29.051,Canal St & Madison St,CHI00498,Wilton Ave & Diversey Pkwy,CHI00226,41.882409,-87.639767,41.932418,-87.652705,member,20.312667,False,False,False,False
5236138,62E7486E7CE580D1,classic_bike,2025-11-15 08:53:42.344,2025-11-15 09:13:49.922,Mies van der Rohe Way & Chicago Ave,CHI00470,Sedgwick St & Schiller St,CHI00303,41.896945,-87.621758,41.907626,-87.638566,member,20.126300,False,False,False,False


In [31]:
# Se inspecciona los datos con duracion negativa

tiempo_invalido=df_muestra[df_muestra['duration_min']<0]
tiempo_invalido.shape
tiempo_invalido

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_min,outlier_2h,outlier_6h,outlier_12h,outlier_24h
5107759,083534D28DA37F72,classic_bike,2025-11-02 01:17:57.001,2025-11-02 01:05:27.752,Clark St & Grace St,CHI00301,Grace St & Central Ave,CHI01799,41.95078,-87.659172,41.949533,-87.767265,member,-12.487483,False,False,False,False
5333210,F87FA50B8D40FA97,electric_bike,2025-11-02 01:54:25.185,2025-11-02 01:10:50.859,NaN,NaN,Michigan Ave & Madison St,CHI01915,41.88000,-87.640000,41.882134,-87.625125,casual,-43.572100,False,False,False,False


In [32]:
#Se guardan los registros con duración negativa en un archivo CSV para trazabilidad y se verifica que se hayan guardado correctamente

#tiempo_invalido.to_csv('tiempo_invalido_muestra.csv', index=False)
tiempo_invalido.shape

(2, 18)

In [33]:
#Se eliminan los registros con duración negativa para tener un conjunto de datos limpio y se verifica que se hayan eliminado correctamente

df_muestra = df_muestra[df_muestra["duration_min"] >= 0]
print(df_muestra.shape)
df_muestra.head()

(99998, 18)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_min,outlier_2h,outlier_6h,outlier_12h,outlier_24h
2021231,AC09C4230F16DC5B,electric_bike,2025-06-30 19:16:08.604,2025-06-30 19:21:26.124,Wolcott Ave & Fargo Ave,CHI01266,Elmwood Ave & Austin St,CHI00739,42.016977,-87.677725,42.025784,-87.684107,casual,5.292000,False,False,False,False
1103055,A38E8273171D67F0,electric_bike,2025-05-18 12:36:15.975,2025-05-18 12:55:40.536,NaN,NaN,Montrose Harbor,TA1308000012,41.900000,-87.620000,41.963982,-87.638181,member,19.409350,False,False,False,False
1834169,0B1F607D321397B0,electric_bike,2025-06-17 20:43:05.503,2025-06-17 21:04:51.961,NaN,NaN,Michigan Ave & Oak St,CHI00252,41.950000,-87.650000,41.900960,-87.623777,member,21.774300,False,False,False,False
277055,21649CA95A4A10B4,electric_bike,2025-02-02 05:04:52.086,2025-02-02 05:08:16.654,NaN,NaN,Clark St & Newport St,632,41.950000,-87.670000,41.944540,-87.654678,casual,3.409467,False,False,False,False
740603,D6CB6A805FC24CD2,classic_bike,2025-04-07 10:24:51.016,2025-04-07 10:31:27.276,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,6.604333,False,False,False,False


Se eliminan los 29 datos que tienen orden temporal invertido ya que  2[tiempo negativo]/100000 [total registros]= 0.002%. 
Según las prácticas de limpieza en análisis de datos <0.1% pueden ser considerados ruido técnico y pueden ser eliminados sin tener impacto. Dado que estos registros representan eventos inválidos desde el punto de vista del negocio, fueron excluidos del dataset analítico. Las columnas temporales se conservaron, y la variable duration_min se utiliza como métrica derivada validada.

In [34]:
df_muestra['duration_min'].describe(percentiles=[0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99])

count    99998.000000
mean        15.979733
std         54.651129
min          0.003400
1%           0.258849
5%           2.067844
10%          3.174378
25%          5.373096
50%          9.437325
75%         16.599417
90%         28.362560
95%         39.825765
99%         88.705005
max       1499.960050
Name: duration_min, dtype: float64

In [35]:
# Análisis de los cuantiles y sus saltos porcentuales para saber cuales outliers tomar en la afectación de las metricas

cuantiles=df_muestra['duration_min'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
saltos_cuantiles=cuantiles.pct_change()*100
saltos_cuantiles

0.25           NaN
0.50     75.640363
0.75     75.891120
0.90     70.864799
0.95     40.416680
0.99    122.732709
Name: duration_min, dtype: float64

In [ ]:
# Media con todos los datos

media_duracion=df_muestra['duration_min'].mean()

#Media excluyendo valores mayores a p99

p99=df_muestra['duration_min'].quantile(0.99) #primero determina el umbral del cuantil 99
sin_p99=df_muestra[df_muestra['duration_min']<=p99] #luego conservan los datos que están por debajo o igual al umbral del cuantil 99
#sin_p99

#Media sin los outliers mayores a p99
media_sin_p99=sin_p99['duration_min'].mean()


#impacto de la media
impacto_media=(media_duracion-media_sin_p99)/media_sin_p99*100

print("Media con todos los datos: ", media_duracion)
print("Media sin los outliers del p99: ", media_sin_p99)   
print("Impacto porcentual de excluir los outliers del p99 en la media: ", round(impacto_media, 2), "%")

Media con todos los datos:  15.979732853990413
Media sin los outliers del p99:  12.941304694034223
Impacto porcentual de excluir los outliers del p99 en la media:  23.48 %


In [42]:
#Mediana con todos los datos

mediana_duracion=df_muestra['duration_min'].median() 

#Mediana excluyendo valores mayores a p99

mediana_sin_p99=sin_p99['duration_min'].median()

#print

print("Mediana con todos los datos: ", mediana_duracion)
print("Mediana sin los outliers del p99: ", mediana_sin_p99)

#Robustez con outliers

robustez_outliers=abs(media_duracion-mediana_duracion)/mediana_duracion*100

#Robustez sin outliers

robustez_sin_outliers=abs(media_sin_p99-mediana_sin_p99)/mediana_sin_p99*100

#print
print("Robustez con outliers: ", round(robustez_outliers, 2), "%")
print("Robustez sin outliers: ", round(robustez_sin_outliers, 2), "%")





Mediana con todos los datos:  9.437325000000001
Mediana sin los outliers del p99:  9.341841666666667
Robustez con outliers:  69.32 %
Robustez sin outliers:  38.53 %


In [43]:
import numpy as np

# Desviación estándar con todos los datos

desviacion_std_duracion=np.std(df_muestra['duration_min'])

# Desviación estándar excluyendo valores mayores a p99

desviacion_std_sin_p99=np.std(sin_p99['duration_min'])

print("Desviación estándar con todos los datos: ", desviacion_std_duracion)
print("Desviación estándar sin los outliers del p99: ", desviacion_std_sin_p99)

# Impacto de los outliers en la desviación estándar
impacto_desviacion_std=(desviacion_std_duracion-desviacion_std_sin_p99)/desviacion_std_sin_p99*100

print("Impacto porcentual de excluir los outliers del p99 en la desviación estándar: ", round(impacto_desviacion_std, 2), "%")


Desviación estándar con todos los datos:  54.65085571242025
Desviación estándar sin los outliers del p99:  11.892090183298428
Impacto porcentual de excluir los outliers del p99 en la desviación estándar:  359.56 %


In [50]:
#Calcular percentiles 99 y 95 con outliers

p95=df_muestra['duration_min'].quantile(0.95)

print(p95,p99)

#Percentiles 99 y 95 sin outliers
p95_sin_outliers=sin_p99['duration_min'].quantile(0.95)
p99_sin_outliers=sin_p99['duration_min'].quantile(0.99)

print("Percentil 95 sin outliers del p99: ", p95_sin_outliers)
print("Percentil 99 sin outliers del p99: ", p99_sin_outliers)

#Impacto de los outliers en los percentiles 95 y 99
impacto_p95=(p95-p95_sin_outliers)/p95_sin_outliers*100
impacto_p99=(p99-p99_sin_outliers)/p99_sin_outliers*100

print("Impacto porcentual de excluir los outliers del p99 en el percentil 95: ", round(impacto_p95, 2), "%")
print("Impacto porcentual de excluir los outliers del p99 en el percentil 99: ", round(impacto_p99, 2), "%")    


39.82576499999999 88.7050053333333
Percentil 95 sin outliers del p99:  36.53919583333333
Percentil 99 sin outliers del p99:  61.5383913333333
Impacto porcentual de excluir los outliers del p99 en el percentil 95:  8.99 %
Impacto porcentual de excluir los outliers del p99 en el percentil 99:  44.15 %


In [51]:
# Indice completo de impacto de los outliers en las métricas centrales (media, mediana, desviación estándar, percentiles)

indice_impacto_outliers = (impacto_media + impacto_desviacion_std + impacto_p99)/3
print("Indice completo de impacto de los outliers en las métricas centrales: ", round(indice_impacto_outliers, 2), "%")


Indice completo de impacto de los outliers en las métricas centrales:  142.39 %
